In [1]:
# ============================================
# Startup Cell: Mount Drive + Load Config + Verify Inputs
# ============================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys

# -------------------------------------------------
# Add project src directory to path
# -------------------------------------------------
SRC_DIR = "/content/drive/MyDrive/DIP_Project/src"
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# -------------------------------------------------
# Import project configuration
# -------------------------------------------------
from project_config import *

TRAIN_FEATURE_VECTORS_CSV = os.path.join(METADATA_DIR, TRAIN_FEATURE_VECTOR_FILENAME)
TEST_FEATURE_VECTORS_CSV  = os.path.join(METADATA_DIR, TEST_FEATURE_VECTOR_FILENAME)

TRAIN_FEATURE_VECTORS_NORMALIZED_CSV = os.path.join(METADATA_DIR, TRAIN_NORMALIZED_FILENAME)
TEST_FEATURE_VECTORS_NORMALIZED_CSV  = os.path.join(METADATA_DIR, TEST_NORMALIZED_FILENAME)

SCALER_OUTPUT_PATH = os.path.join(MODELS_DIR, SCALER_FILENAME)

print("Project configuration loaded.")
print()

# -------------------------------------------------
# Verify required input files exist
# -------------------------------------------------
TRAIN_FEATURE_VECTORS_CSV = os.path.join(METADATA_DIR, TRAIN_FEATURE_VECTOR_FILENAME)
TEST_FEATURE_VECTORS_CSV  = os.path.join(METADATA_DIR, TEST_FEATURE_VECTOR_FILENAME)

REQUIRED_FILES = [
    TRAIN_FEATURE_VECTORS_CSV,
    TEST_FEATURE_VECTORS_CSV
]

TRAIN_FEATURE_VECTORS_NORMALIZED_CSV = os.path.join(METADATA_DIR, TRAIN_NORMALIZED_FILENAME)
TEST_FEATURE_VECTORS_NORMALIZED_CSV  = os.path.join(METADATA_DIR, TEST_NORMALIZED_FILENAME)

SCALER_OUTPUT_PATH = os.path.join(MODELS_DIR, SCALER_FILENAME)

print("Verifying required input files...\n")

for file_path in REQUIRED_FILES:
    if os.path.exists(file_path):
        print(f"{file_path}  --> OK")
    else:
        raise FileNotFoundError(f"Missing required file: {file_path}")

print("\nAll required input files are present.")



Mounted at /content/drive
Project configuration loaded.

Verifying required input files...

/content/drive/MyDrive/DIP_Project/data/metadata/train_feature_vectors.csv  --> OK
/content/drive/MyDrive/DIP_Project/data/metadata/test_feature_vectors.csv  --> OK

All required input files are present.


In [2]:
# ============================================
# Cell 0: Notebook Overview
# ============================================
#
# Purpose:
# Normalize DIP feature vectors using training-set statistics
# only and prepare classifier-ready inputs. Outputs include
# normalized train/test CSV files and a saved scaler for
# reuse in model training.
#
# Inputs:
# - train_feature_vectors.csv
# - test_feature_vectors.csv
# - project_config.py
#
# Outputs:
# - train_feature_vectors_normalized.csv
# - test_feature_vectors_normalized.csv
# - scaler.pkl
#
# Assumptions:
# - Notebook 05 has already generated the train/test feature vector CSV files.
# - The dataset uses a train/test split only (no validation set).
# - Metadata columns are preserved and are not normalized.
# - Only numeric DIP feature columns are normalized.
# - The scaler is fit on training data only and applied to both train and test.
#
# What the notebook does:
#   Startup Cell:
#     Mount Google Drive, import project_config.py, and verify that
#     required train/test feature-vector CSV files exist.
#
#   Cell 1:
#     Import required libraries for data handling, normalization,
#     and saving artifacts.
#
#   Cell 2:
#     Load the training and test feature-vector CSV files and display
#     basic shape and structure information.
#
#   Cell 3:
#     Define metadata and feature column groupings and confirm that
#     train and test datasets have consistent structure.
#
#   Cell 4:
#     Validate inputs:
#       - confirm train/test column alignment
#       - confirm expected feature count
#       - confirm no missing values in feature columns
#
#   Cell 5:
#     Split each dataset into:
#       - metadata tables
#       - numeric feature matrices for normalization
#
#   Cell 6:
#     Fit a StandardScaler using the training feature matrix only
#     to prevent data leakage.
#
#   Cell 7:
#     Apply the fitted scaler to both training and test feature
#     matrices and convert results back to DataFrames.
#
#   Cell 8:
#     Recombine metadata with normalized feature columns to form
#     final normalized datasets.
#
#   Cell 9:
#     Save:
#       - train_feature_vectors_normalized.csv
#       - test_feature_vectors_normalized.csv
#       - scaler.pkl
#
#   Cell 10:
#     Perform sanity checks on normalized outputs:
#       - confirm shapes
#       - inspect sample rows
#       - verify normalization behavior (mean ~0, std ~1 for train set)
#
# Notes:
#   - Standard deviation values in the training set may not appear
#     exactly as 1.0 due to the use of sample standard deviation
#     (ddof=1) in pandas. Values should be very close to 1.0.
#
# ============================================

print("Notebook overview loaded.")



Notebook overview loaded.


In [3]:
# ============================================
# Cell 1: Import Required Libraries
# ============================================

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler



In [4]:
# ============================================
# Cell 2: Load Train and Test Feature Vectors
# ============================================

# Load CSV files (paths already defined in Startup Cell)
df_train = pd.read_csv(TRAIN_FEATURE_VECTORS_CSV)
df_test  = pd.read_csv(TEST_FEATURE_VECTORS_CSV)

print("Train and test feature vectors loaded.")
print()

print("df_train shape:", df_train.shape)
print("df_test shape: ", df_test.shape)

print("\nColumns:")
for col in df_train.columns:
    print(col)



Train and test feature vectors loaded.

df_train shape: (14400, 29)
df_test shape:  (3600, 29)

Columns:
filename
class_label
source_dataset
subset
Mean Gradient
Std Gradient
Max Gradient
Gradient Entropy
Edge Density
Orientation Mean
Orientation Std
Orientation Entropy
Global Entropy
Local Entropy Mean
Local Entropy Std
Intensity Mean
Intensity Std
Laplacian Variance
Patch Variance Mean
Patch Variance Std
Noise Residual Energy
Low Frequency Energy Ratio
High Frequency Energy Ratio
Radial Mean
Radial Std
Radial Entropy
Spectral Centroid
Spectral Bandwidth
Log Spectrum Std


In [5]:
# ============================================
# Cell 3: Load Train and Test Feature Vectors
# ============================================

df_train = pd.read_csv(TRAIN_FEATURE_VECTORS_CSV)
df_test  = pd.read_csv(TEST_FEATURE_VECTORS_CSV)

print("Train and test feature vectors loaded.")
print()
print("df_train shape:", df_train.shape)
print("df_test shape: ", df_test.shape)



Train and test feature vectors loaded.

df_train shape: (14400, 29)
df_test shape:  (3600, 29)


In [6]:
# ============================================
# Cell 4: Validate Input Structure
# ============================================

feature_columns = [col for col in df_train.columns if col not in METADATA_COLUMNS]

assert list(df_train.columns) == list(df_test.columns), \
    "Train and test columns do not match."

for col in METADATA_COLUMNS:
    assert col in df_train.columns, f"Missing metadata column: {col}"

assert len(feature_columns) == NUM_FEATURES, \
    f"Expected {NUM_FEATURES} feature columns, found {len(feature_columns)}."

assert not df_train[feature_columns].isnull().any().any(), \
    "Missing values found in training feature columns."

assert not df_test[feature_columns].isnull().any().any(), \
    "Missing values found in test feature columns."

print("Input validation passed.")
print()
print("Metadata columns:")
for col in METADATA_COLUMNS:
    print(" ", col)

print("\nFeature column count:", len(feature_columns))



Input validation passed.

Metadata columns:
  filename
  class_label
  source_dataset
  subset

Feature column count: 25


In [7]:
# ============================================
# Cell 5: Split Metadata and Feature Matrices
# ============================================

assert len(feature_columns) == 25, f"Expected 25 feature columns, found {len(feature_columns)}."

# Separate metadata columns
train_meta = df_train[METADATA_COLUMNS].copy()
test_meta = df_test[METADATA_COLUMNS].copy()

# Separate numeric feature columns
X_train = df_train[feature_columns].copy()
X_test = df_test[feature_columns].copy()

print("Metadata and feature matrices created.")
print()

print("train_meta shape:", train_meta.shape)
print("test_meta shape: ", test_meta.shape)
print("X_train shape:    ", X_train.shape)
print("X_test shape:     ", X_test.shape)

print("\nFirst 5 metadata rows (train):")
display(train_meta.head())

print("\nFirst 5 feature rows (train):")
display(X_train.head())



Metadata and feature matrices created.

train_meta shape: (14400, 4)
test_meta shape:  (3600, 4)
X_train shape:     (14400, 25)
X_test shape:      (3600, 25)

First 5 metadata rows (train):


,filename,class_label,source_dataset,subset
0,rl_imgn_002320.png,rl,ImageNet_1K_256,train
1,rl_coco_001397.png,rl,MS_COCO_2017,train
2,rl_imgn_001958.png,rl,ImageNet_1K_256,train
3,rl_coco_000800.png,rl,MS_COCO_2017,train
4,ai_mj_002892.png,ai,Midjourney,train



First 5 feature rows (train):


,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,Orientation Std,Orientation Entropy,Global Entropy,Local Entropy Mean,...,Patch Variance Std,Noise Residual Energy,Low Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,113.284256,148.626465,988.380493,4.200053,0.133926,0.009213,1.708909,4.952383,7.730578,2.483552,...,1845.180777,51.161957,0.977534,0.005830,4.172419e+11,5.524948e+12,0.049157,0.042217,0.785720,0.944199
1,58.244781,91.992287,943.153198,3.224193,0.104218,0.003133,1.812218,5.067689,7.022424,1.772827,...,1525.370321,26.302368,0.975984,0.004450,8.717484e+10,1.120272e+12,0.098270,0.103196,0.981119,1.096229
2,99.089622,129.458405,955.745789,4.071966,0.119492,-0.031641,1.761121,5.136969,7.350558,2.544420,...,1353.469364,47.263107,0.970208,0.007792,2.700694e+11,3.598769e+12,0.049157,0.033637,0.839125,0.934185
3,76.498199,114.998993,924.299744,3.541116,0.123779,0.210707,1.640483,5.021997,7.337275,1.882955,...,1453.744813,35.510101,0.992152,0.001953,6.987824e+11,9.309489e+12,0.049157,0.018659,0.438470,0.960530
4,67.460518,74.748627,801.519836,3.823527,0.113922,-0.004285,1.853334,5.145963,7.745233,2.290586,...,971.494479,27.173721,0.993974,0.000909,3.318048e+11,4.394753e+12,0.049157,0.035712,0.498081,1.125460


In [8]:
# ============================================
# Cell 6: Fit Scaler on Training Features
# ============================================

from sklearn.preprocessing import StandardScaler
import joblib

# Initialize scaler
scaler = StandardScaler()

# Fit scaler using training features only
scaler.fit(X_train)

print("Scaler fitted on training feature matrix only.")
print("This scaler will be applied to both train and test data.")
print()

print("Number of features learned by scaler:", len(scaler.mean_))
print("\nFirst 5 feature means learned from training data:")
for col, val in zip(feature_columns[:5], scaler.mean_[:5]):
    print(f"  {col}: {val:.6f}")

print("\nFirst 5 feature standard deviations learned from training data:")
for col, val in zip(feature_columns[:5], scaler.scale_[:5]):
    print(f"  {col}: {val:.6f}")



Scaler fitted on training feature matrix only.
This scaler will be applied to both train and test data.

Number of features learned by scaler: 25

First 5 feature means learned from training data:
  Mean Gradient: 75.671611
  Std Gradient: 100.890452
  Max Gradient: 836.347525
  Gradient Entropy: 3.600815
  Edge Density: 0.115457

First 5 feature standard deviations learned from training data:
  Mean Gradient: 38.647885
  Std Gradient: 39.507917
  Max Gradient: 166.424395
  Gradient Entropy: 0.771241
  Edge Density: 0.025125


In [9]:
# ============================================
# Cell 7: Transform Train and Test Features
# ============================================

# Apply the fitted scaler to train and test feature matrices
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames
df_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_columns)
df_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_columns)

print("Training and test feature matrices normalized.")
print()

print("df_train_scaled shape:", df_train_scaled.shape)
print("df_test_scaled shape: ", df_test_scaled.shape)

print("\nFirst 5 normalized feature rows (train):")
display(df_train_scaled.head())



Training and test feature matrices normalized.

df_train_scaled shape: (14400, 25)
df_test_scaled shape:  (3600, 25)

First 5 normalized feature rows (train):


,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,Orientation Std,Orientation Entropy,Global Entropy,Local Entropy Mean,...,Patch Variance Std,Noise Residual Energy,Low Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,0.973214,1.208264,0.913526,0.776979,0.735101,-0.207859,-0.423789,0.017518,0.787559,0.805812,...,1.019527,1.071297,-0.189219,0.417247,0.448644,0.439427,-0.549996,-0.084929,0.290945,-1.254194
1,-0.450913,-0.225225,0.641767,-0.488332,-0.447341,-0.246587,0.400435,0.345097,-0.180400,-0.369032,...,0.509586,-0.330229,-0.264290,0.144170,-1.327122,-1.325683,1.397134,0.856623,0.808074,0.071118
2,0.605933,0.723094,0.717432,0.610900,0.160581,-0.468100,-0.007225,0.541920,0.268118,0.906429,...,0.235488,0.851489,-0.544033,0.805334,-0.343146,-0.332461,-0.549996,-0.217414,0.432281,-1.341491
3,0.021388,0.357107,0.528482,-0.077406,0.331237,1.075664,-0.969709,0.215289,0.249962,-0.186989,...,0.395378,0.188882,0.518813,-0.349898,1.963337,1.956027,-0.549996,-0.448685,-0.628062,-1.111831
4,-0.212459,-0.661686,-0.209270,0.288772,-0.061089,-0.293840,0.728471,0.567472,0.807590,0.486835,...,-0.373574,-0.281104,0.607047,-0.556607,-0.011009,-0.013482,-0.549996,-0.185376,-0.470299,0.325939


In [10]:
# ============================================
# Cell 8: Recombine Metadata with Normalized Features
# ============================================

# Reset indices to ensure clean horizontal concatenation
train_meta = train_meta.reset_index(drop=True)
test_meta = test_meta.reset_index(drop=True)
df_train_scaled = df_train_scaled.reset_index(drop=True)
df_test_scaled = df_test_scaled.reset_index(drop=True)

# Recombine metadata and normalized feature columns
df_train_normalized = pd.concat([train_meta, df_train_scaled], axis=1)
df_test_normalized = pd.concat([test_meta, df_test_scaled], axis=1)

print("Metadata recombined with normalized feature columns.")
print()

print("df_train_normalized shape:", df_train_normalized.shape)
print("df_test_normalized shape: ", df_test_normalized.shape)

print("\nFirst 5 rows of normalized training table:")
display(df_train_normalized.head())



Metadata recombined with normalized feature columns.

df_train_normalized shape: (14400, 29)
df_test_normalized shape:  (3600, 29)

First 5 rows of normalized training table:


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Patch Variance Std,Noise Residual Energy,Low Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,rl_imgn_002320.png,rl,ImageNet_1K_256,train,0.973214,1.208264,0.913526,0.776979,0.735101,-0.207859,...,1.019527,1.071297,-0.189219,0.417247,0.448644,0.439427,-0.549996,-0.084929,0.290945,-1.254194
1,rl_coco_001397.png,rl,MS_COCO_2017,train,-0.450913,-0.225225,0.641767,-0.488332,-0.447341,-0.246587,...,0.509586,-0.330229,-0.264290,0.144170,-1.327122,-1.325683,1.397134,0.856623,0.808074,0.071118
2,rl_imgn_001958.png,rl,ImageNet_1K_256,train,0.605933,0.723094,0.717432,0.610900,0.160581,-0.468100,...,0.235488,0.851489,-0.544033,0.805334,-0.343146,-0.332461,-0.549996,-0.217414,0.432281,-1.341491
3,rl_coco_000800.png,rl,MS_COCO_2017,train,0.021388,0.357107,0.528482,-0.077406,0.331237,1.075664,...,0.395378,0.188882,0.518813,-0.349898,1.963337,1.956027,-0.549996,-0.448685,-0.628062,-1.111831
4,ai_mj_002892.png,ai,Midjourney,train,-0.212459,-0.661686,-0.209270,0.288772,-0.061089,-0.293840,...,-0.373574,-0.281104,0.607047,-0.556607,-0.011009,-0.013482,-0.549996,-0.185376,-0.470299,0.325939


In [11]:
# ============================================
# Cell 9: Save Normalized Feature Vectors and Scaler
# ============================================

import os
import joblib

# Ensure output directories exist
os.makedirs(os.path.dirname(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV), exist_ok=True)
os.makedirs(os.path.dirname(TEST_FEATURE_VECTORS_NORMALIZED_CSV), exist_ok=True)
os.makedirs(os.path.dirname(SCALER_OUTPUT_PATH), exist_ok=True)

# Save normalized feature vector CSVs
df_train_normalized.to_csv(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV, index=False)
df_test_normalized.to_csv(TEST_FEATURE_VECTORS_NORMALIZED_CSV, index=False)

# Save fitted scaler
joblib.dump(scaler, SCALER_OUTPUT_PATH)

print("Saved normalized feature vectors and scaler.")
print()

print("Train normalized CSV:", TRAIN_FEATURE_VECTORS_NORMALIZED_CSV)
print("Test normalized CSV: ", TEST_FEATURE_VECTORS_NORMALIZED_CSV)
print("Scaler saved to:     ", SCALER_OUTPUT_PATH)



Saved normalized feature vectors and scaler.

Train normalized CSV: /content/drive/MyDrive/DIP_Project/data/metadata/train_feature_vectors_normalized.csv
Test normalized CSV:  /content/drive/MyDrive/DIP_Project/data/metadata/test_feature_vectors_normalized.csv
Scaler saved to:      /content/drive/MyDrive/DIP_Project/models/scaler.pkl


In [12]:
# ============================================
# Cell 10: Sanity Check Normalized Outputs
# ============================================

# Identify normalized feature columns
normalized_feature_columns = [col for col in df_train_normalized.columns if col not in METADATA_COLUMNS]

# Compute summary statistics for normalized training features
train_feature_means = df_train_normalized[normalized_feature_columns].mean()
train_feature_stds = df_train_normalized[normalized_feature_columns].std()

# Compute summary statistics for normalized test features
test_feature_means = df_test_normalized[normalized_feature_columns].mean()
test_feature_stds = df_test_normalized[normalized_feature_columns].std()

print("Sanity check on normalized outputs:")
print()

print("Train normalized shape:", df_train_normalized.shape)
print("Test normalized shape: ", df_test_normalized.shape)

print("\nFirst 5 rows of normalized training data:")
display(df_train_normalized.head())

print("\nTraining feature mean summary (should be near 0):")
print(train_feature_means.round(6))

print("\nTraining feature std summary (should be near 1):")
print(train_feature_stds.round(6))

print("\nTest feature mean summary (not necessarily 0):")
print(test_feature_means.round(6))

print("\nTest feature std summary (not necessarily 1):")
print(test_feature_stds.round(6))



Sanity check on normalized outputs:

Train normalized shape: (14400, 29)
Test normalized shape:  (3600, 29)

First 5 rows of normalized training data:


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Patch Variance Std,Noise Residual Energy,Low Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,rl_imgn_002320.png,rl,ImageNet_1K_256,train,0.973214,1.208264,0.913526,0.776979,0.735101,-0.207859,...,1.019527,1.071297,-0.189219,0.417247,0.448644,0.439427,-0.549996,-0.084929,0.290945,-1.254194
1,rl_coco_001397.png,rl,MS_COCO_2017,train,-0.450913,-0.225225,0.641767,-0.488332,-0.447341,-0.246587,...,0.509586,-0.330229,-0.264290,0.144170,-1.327122,-1.325683,1.397134,0.856623,0.808074,0.071118
2,rl_imgn_001958.png,rl,ImageNet_1K_256,train,0.605933,0.723094,0.717432,0.610900,0.160581,-0.468100,...,0.235488,0.851489,-0.544033,0.805334,-0.343146,-0.332461,-0.549996,-0.217414,0.432281,-1.341491
3,rl_coco_000800.png,rl,MS_COCO_2017,train,0.021388,0.357107,0.528482,-0.077406,0.331237,1.075664,...,0.395378,0.188882,0.518813,-0.349898,1.963337,1.956027,-0.549996,-0.448685,-0.628062,-1.111831
4,ai_mj_002892.png,ai,Midjourney,train,-0.212459,-0.661686,-0.209270,0.288772,-0.061089,-0.293840,...,-0.373574,-0.281104,0.607047,-0.556607,-0.011009,-0.013482,-0.549996,-0.185376,-0.470299,0.325939



Training feature mean summary (should be near 0):
Mean Gradient                 -0.0
Std Gradient                   0.0
Max Gradient                   0.0
Gradient Entropy               0.0
Edge Density                  -0.0
Orientation Mean               0.0
Orientation Std                0.0
Orientation Entropy           -0.0
Global Entropy                -0.0
Local Entropy Mean             0.0
Local Entropy Std              0.0
Intensity Mean                 0.0
Intensity Std                 -0.0
Laplacian Variance            -0.0
Patch Variance Mean            0.0
Patch Variance Std             0.0
Noise Residual Energy         -0.0
Low Frequency Energy Ratio    -0.0
High Frequency Energy Ratio   -0.0
Radial Mean                   -0.0
Radial Std                     0.0
Radial Entropy                -0.0
Spectral Centroid             -0.0
Spectral Bandwidth            -0.0
Log Spectrum Std               0.0
dtype: float64

Training feature std summary (should be near 1):
Mean Grad